# LieselOptim Showcase

`LieselOptim` is the opinionated builder for optimizing a Liesel model. It creates a split, loss, batches, optimizer, stopper, and engine using compact defaults, while still returning an `OptimEngine` for further configuration.

In [1]:
import jax
import jax.numpy as jnp
import optax
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl
import liesel.optim as opt


def normal_model(n=32, *, seed=1, name="y"):
    key = jax.random.key(seed)
    y_values = 1.4 + 0.35 * jax.random.normal(key, (n,))
    loc = lsl.Var.new_param(jnp.array(0.0), name="loc")
    y = lsl.Var.new_obs(
        y_values,
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name=name,
    )
    return lsl.Model([y])


def two_branch_model(*, seed=10):
    key1, key2 = jax.random.split(jax.random.key(seed))
    loc = lsl.Var.new_param(jnp.array(0.0), name="loc")
    y1_values = 1.2 + 0.25 * jax.random.normal(key1, (30,))
    y2_values = -0.6 + 0.40 * jax.random.normal(key2, (11,))
    y1 = lsl.Var.new_obs(
        y1_values,
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name="y_long",
    )
    y2 = lsl.Var.new_obs(
        y2_values,
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name="y_short",
    )
    return lsl.Model([y1, y2])

## Default Builder Configuration

By default, `LieselOptim` builds a scaled negative log posterior loss, full-data batches, and an Adam optimizer.

In [2]:
model = normal_model(seed=1)
builder = opt.LieselOptim(model, stopper=opt.Stopper(epochs=5, patience=5), seed=1)
engine = builder.build_engine()

{
    "engine_type": type(engine).__name__,
    "loss_type": type(engine.loss).__name__,
    "loss_scaled": engine.loss.scale,
    "batch_type": type(engine.batches).__name__,
    "optimizer_ids": [optimizer.identifier for optimizer in engine.optimizers],
}

{'engine_type': 'OptimEngine',
 'loss_type': 'NegLogProbLoss',
 'loss_scaled': True,
 'batch_type': 'Batches',
 'optimizer_ids': ['000']}

## Short Fit

For notebook-friendly output, build the engine, turn off the progress bar, and call `fit()`.

In [3]:
quick_engine = opt.LieselOptim(
    normal_model(seed=2),
    stopper=opt.Stopper(epochs=5, patience=5),
    seed=2,
).build_engine()
quick_result = quick_engine.fit()

{
    "result_type": type(quick_result).__name__,
    "final_epoch": quick_result.final_epoch,
    "best_loc": round(float(quick_result.best_position["loc"]), 4),
}

Initializing:   0%|          | 0/5 [00:00<?, ?it/s]

Initializing:  20%|██        | 1/5 [00:00<00:00,  5.33it/s]

Training loss: 1.929, Monitoring loss: 1.929:  20%|██        | 1/5 [00:00<00:00,  5.33it/s]

Training loss: 1.928, Monitoring loss: 1.928:  40%|████      | 2/5 [00:00<00:00,  5.33it/s]

Training loss: 1.926, Monitoring loss: 1.926:  60%|██████    | 3/5 [00:00<00:00,  5.33it/s]

Training loss: 1.925, Monitoring loss: 1.925:  80%|████████  | 4/5 [00:00<00:00,  5.33it/s]

Training loss: 1.924, Monitoring loss: 1.924: 100%|██████████| 5/5 [00:00<00:00,  5.33it/s]

Training loss: 1.924, Monitoring loss: 1.924: 100%|██████████| 5/5 [00:00<00:00,  5.33it/s]

Training loss: 1.924, Monitoring loss: 1.924: 100%|██████████| 5/5 [00:00<00:00, 22.44it/s]

{'result_type': 'OptimResult', 'final_epoch': 5, 'best_loc': 0.005}

## Mini-Batch Training

Passing `batch_axis_size` asks the builder to construct training batches from the resolved split.

In [4]:
mini_engine = opt.LieselOptim(
    normal_model(seed=3),
    batch_axis_size=8,
    stopper=opt.Stopper(epochs=5, patience=5),
    seed=3,
).build_engine()

{
    "batch_type": type(mini_engine.batches).__name__,
    "axis_size": mini_engine.batches.axis_size,
    "batch_axis_size": mini_engine.batches.batch_axis_size,
    "n_full_batches": mini_engine.batches.n_full_batches,
}

{'batch_type': 'Batches',
 'axis_size': 32,
 'batch_axis_size': 8,
 'n_full_batches': 4}

## Validation Split And Train Monitor

When a validation split is provided, it is used by the loss and engine. The train monitor controls how training loss is summarized between validation checks.

In [5]:
valid_model = normal_model(seed=4)
valid_split = opt.PositionSplit.from_model(
    valid_model,
    validate_axis_share=0.25,
    shuffle=True,
    seed=4,
)
valid_engine = opt.LieselOptim(
    valid_model,
    split=valid_split,
    batch_axis_size=8,
    train_monitor="weighted_epoch_average",
    stopper=opt.Stopper(epochs=5, patience=5),
    seed=4,
).build_engine()

{
    "has_validation": valid_engine.split.has_validation,
    "train_axis_size": valid_engine.split.train_axis_size,
    "validate_axis_size": valid_engine.split.validate_axis_size,
    "train_monitor": valid_engine.train_monitor,
}

{'has_validation': True,
 'train_axis_size': 24,
 'validate_axis_size': 8,
 'train_monitor': 'weighted_epoch_average'}

## Multi-Size Observed Branches

If observed variables have different sample sizes, the builder opts into manager objects automatically.

In [6]:
multi_engine = opt.LieselOptim(
    two_branch_model(seed=5),
    batch_axis_size=6,
    stopper=opt.Stopper(epochs=4, patience=4),
    seed=5,
).build_engine()

{
    "split_type": type(multi_engine.split).__name__,
    "batch_type": type(multi_engine.batches).__name__,
    "train_axis_sizes": multi_engine.split.train_axis_sizes,
    "n_full_batches": multi_engine.batches.n_full_batches,
}

{'split_type': 'PositionSplitManager',
 'batch_type': 'BatchManager',
 'train_axis_sizes': (11, 30),
 'n_full_batches': 5}

## Customization Examples

You can disable loss scaling, pass explicit optimizers, and still inspect or modify the engine before fitting.

In [7]:
custom_model = normal_model(seed=6)
custom_optimizer = opt.Optimizer(
    list(custom_model.parameters),
    optax.sgd(0.01),
    identifier="sgd",
)
custom_engine = opt.LieselOptim(
    custom_model,
    scale_loss=False,
    optimizers=[custom_optimizer],
    stopper=opt.Stopper(epochs=3, patience=3),
    seed=6,
).build_engine()
custom_result = custom_engine.fit()

{
    "loss_scaled": custom_engine.loss.scale,
    "optimizer_ids": [optimizer.identifier for optimizer in custom_engine.optimizers],
    "final_epoch": custom_result.final_epoch,
}

Initializing:   0%|          | 0/3 [00:00<?, ?it/s]

Initializing:  33%|███▎      | 1/3 [00:00<00:00,  7.49it/s]

Training loss: 44.106, Monitoring loss: 44.106:  33%|███▎      | 1/3 [00:00<00:00,  7.49it/s]

Training loss: 36.794, Monitoring loss: 36.794:  67%|██████▋   | 2/3 [00:00<00:00,  7.49it/s]

Training loss: 33.413, Monitoring loss: 33.413: 100%|██████████| 3/3 [00:00<00:00,  7.49it/s]

Training loss: 33.413, Monitoring loss: 33.413: 100%|██████████| 3/3 [00:00<00:00,  7.49it/s]

Training loss: 33.413, Monitoring loss: 33.413: 100%|██████████| 3/3 [00:00<00:00, 21.99it/s]

{'loss_scaled': False, 'optimizer_ids': ['sgd'], 'final_epoch': 3}